# Batchsimulation mit Python

## Voraussetzungen

Windows mit Microsoft Excel, eine gespeicherte PEExcel-Arbeitskopie und vorbereitete Szenarien.
Installiere im Repository `python -m pip install -e ".[batch]"`.
Die Arbeitsmappe benötigt `RunScenariosAutomated(reloadBase As Boolean) As Boolean`:
VBA lädt optional die Basis, wendet Presets an, berechnet und aktualisiert den Status.
VBA speichert und exportiert dabei nicht. Python erfasst IN/STAGE und OUT/Live sofort.
Für Stundenwerte wird zusätzlich die strukturierte Tabelle SIM gelesen.

Dieses Tutorial verwendet `Project` und die öffentliche API `pexl.batch.run_batch`.
Es benötigt keine direkten Excel-Automationsaufrufe. Die Berechnung ist zunächst ausgeschaltet;
zum Ausführen Pfade prüfen und `RUN_EXCEL = True` setzen. Vorher die Arbeitskopie speichern.
Während des Laufs keine Eingaben in Excel ändern.

### Aufgabenteilung in pexl

`pexl.batch.run_batch` organisiert den Batch-Auftrag: Szenarienauswahl und Reihenfolge,
Ergebnisdateien, Fortschritt und Wiederaufnahme. `Project` und `Scenario` enthalten die Daten.

Der allgemeine `pexl.io.excel_com.PEExcelAdapter` übernimmt die Verbindung zur Arbeitsmappe
und den Zugriff auf Excel-Tabellen. Szenarien lesen, konfigurieren, ausführen und aktuelle
Ergebnisse erfassen sind seine bisher implementierten Operationen. Die Verbindung selbst
setzt keine SCEN-Tabelle voraus; Tabellen werden erst bei Bedarf angesprochen. So können
künftige PEExcel-Funktionen dieselbe Adapter-Infrastruktur verwenden.

Für dieses Tutorial genügt `run_batch`; ein direkter Adapter- oder COM-Aufruf ist nicht nötig.


In [ ]:
from pathlib import Path
import pandas as pd
from pexl import Project
from pexl.batch import run_batch

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pexl").is_dir())
workbook = ROOT / "excel/development/ka_PEQ_Nachweistool_DEV.xlsb"
output_dir = ROOT / "notebooks/local_exports/batch_api"
RUN_EXCEL = False
scenario_source = None  # None: vorhandene SCEN-Tabelle; alternativ Pfad zu CSV/XLSX
result = None

## 1. Szenarien aus Excel oder einer externen Liste

Ohne `scenario_source` werden die Zeilen der vorhandenen Tabelle `Scenarios` auf SCEN verwendet.
Die API wählt anhand des Filters aus; die bisherige Spalte `Simulieren?` begrenzt den Lauf nicht.
Pro VBA-Aufruf wird genau ein Szenario ausgewählt.

Alternativ eine CSV/XLSX mit denselben SCEN-Spaltennamen angeben. Pflichtfelder sind
`Scenario Name` und `Base Scenario`, dazu kommen die gewünschten Preset-Spalten mit ihren
exakten Namen aus der Arbeitsmappe. Unbekannte Spalten werden abgewiesen. Jede Zeile braucht
einen eindeutigen Szenarionamen. Die API schreibt die Liste temporär nach SCEN und stellt
anschließend die ursprüngliche Konfiguration einschließlich Formeln wieder her.

Eine Excel-Liste verwendet das Blatt `Scenarios` oder `SCEN`, alternativ genau ein Blatt.
CSV-Dateien besitzen eine Kopfzeile. Beispielkonfiguration (Pfad an eigene Liste anpassen):

```python
scenario_source = ROOT / "data/archetypes.csv"
# oder: scenario_source = ROOT / "data/archetypes.xlsx"
# oder direkt: scenario_source = pd.DataFrame(...)
```

Für den ersten Live-Test nur 2–5 Szenarien verwenden. Ein eigener Ausgabeordner je
Workbook-/Konfigurationsstand verhindert die Vermischung unterschiedlicher Modellstände.

Auswahl, Save-/Exportflags und `Skip base reload?` werden durch die API gesteuert;
die entsprechenden Listeneinträge werden dafür vorübergehend überschrieben.
`Zeitpunkt` und Statusfelder gehören nicht zur Resume-Eingabeidentität.

## 2. Jahresergebnisse berechnen und sichern

`batch_size=25` bedeutet höchstens 25 Szenarien pro Jahresergebnisdatei, nicht 25 gleichzeitige
Excel-Berechnungen. Für ungefähr 5.000 Typologien entstehen etwa 200 Jahresdateien.
Stundenwerte bleiben für diesen Anwendungsfall ausgeschaltet.

`base_reload="always"` lädt die Basis vor jedem Szenario und verhindert übernommene Eingaben.
`"first_only"` ist schneller, setzt aber vollständige, unabhängige Presets voraus: die Basis
wird beim ersten ausgeführten Szenario und bei einem Wechsel der Basis geladen, auch nach Resume.
Unvollständige Presets können sonst Werte aus dem vorherigen Szenario übernehmen.

In [ ]:
if RUN_EXCEL:
    result = run_batch(
        workbook=workbook,
        scenarios=scenario_source,
        scenario_filter={"Base Scenario": "Default"},
        batch_size=25,
        base_reload="always",
        export_annual=True,
        export_timeseries=False,
        output_dir=output_dir,
        resume=True,
    )
    print("Status:", result.status)
    print("Abgeschlossen:", len(result.completed), "bereits vorhanden:", len(result.skipped))
    print("Fehlgeschlagen:", result.failed)
    print("Dateien:", result.exported_files)

## 3. Jahresdaten als Project konsolidieren

Die Jahresdateien enthalten IN, OUT und META im bestehenden Projektformat. Es gibt keinen
separaten Batch-Schema-Mapper. Für die Weiterverarbeitung im Gebäudesektormodell können die
Dateien mit `Project.from_files` zusammengeführt werden. Nur bestätigte Dateien aus dem
Ergebnis verwenden, nicht pauschal alle Dateien eines nach Abbruch verbliebenen Ordners.

Für sehr große Aufträge lassen sich dieselben Dateien einzeln auswerten, um Speicher zu sparen.

In [ ]:
annual_files = []
project = None
if result is not None:
    annual_files = [p for p in result.exported_files if p.suffix == ".xlsx"]
    if annual_files:
        project = Project.from_files(annual_files, unknown="raise")
        assert project.column_names() == result.completed
        summary = pd.DataFrame([
            {"scenario": s.column_name, "QH": s.to_var_dict().get("QH")}
            for s in project
        ])
        display(summary.head())

## 4. Unterbrechen und fortsetzen

Denselben Aufruf mit `resume=True` erneut ausführen. Die API prüft Workbook-Dateihash,
SCEN-Definitionen, Schema, Optionen und Hashes der Ergebnisdateien. Abweichungen führen zu
einem Fehler; dafür einen neuen Ausgabeordner verwenden. Vor Beginn den gewünschten Excel-Basisstand
speichern: ungespeicherte Änderungen außerhalb von SCEN werden vom Dateihash nicht erfasst.

Bei einem VBA-/Capture-Fehler stoppt der Lauf; `result.failed` erklärt den betroffenen Eintrag.
Bereits erfasste Ergebnisse werden gesichert. Setup- und Schreibfehler werden als Exceptions
weitergegeben. Nach einem harten Abbruch wird der letzte unbestätigte Chunk erneut gerechnet.
Ein verbliebenes `.batch.lock` erst entfernen, wenn sicher kein Excel-Lauf mehr aktiv ist.

Fremde Excel-Instanzen werden nicht beendet, die Arbeitsmappe wird nicht gespeichert.
Eine selbst geöffnete Arbeitsmappe wird ohne Speichern geschlossen. Nach einem harten
Prozessabbruch ist eine Wiederherstellung von SCEN nicht garantiert; deshalb Arbeitskopien nutzen.
Der VBA-Aufruf läuft synchron und hat keinen erzwungenen Timeout.

## 5. Optionale Stundenwerte

Für gezielte Kontrollszenarien `export_timeseries=True` und einen separaten Ausgabeordner setzen.
Die API erfasst SIM über `ScenarioTimeseries` und schreibt pro Szenario eine CSV. Die Zuordnung
steht im Manifest `batch.json`; numerische Dateinamen vermeiden Konflikte durch Sonderzeichen.
Die Rohdaten behalten die SIM-Spaltennamen. Einheiten stehen im Zeitreihenschema.

```python
# Nach einem Lauf mit export_timeseries=True:
series_files = [p for p in result.exported_files if p.suffix == ".csv"]
series = pd.read_csv(series_files[0])
print(series.shape, series.columns.tolist())
```

Jahresergebnisse werden mit `read_project` bzw. `Project.from_files` gelesen. Die Stunden-CSV
wird separat gelesen und kann bei Bedarf mit `scenario.set_timeseries(series)` angehängt werden.
Eine Zahl von 8.760 Stunden ist keine universelle Validierung für jedes Kalenderjahr.

## Weiterarbeiten

Das [Python-Angebot](https://simonschaluppe.github.io/peexcel/python.html) zeigt Analysen und
Visualisierungen mit bestehenden Project-Views. Der
[Parameter- und Kennzahlenkatalog](https://simonschaluppe.github.io/peexcel/glossary.html)
erläutert die verfügbaren Variablen.

[Dieses Tutorial als Notebook herunterladen](downloads/tutorial_batch.ipynb).